```text
Qt 里的 worker-thread 退出机制，核心要分清楚三个对象：

1. QThread 对象：一个 QObject，通常活在主线程里，用来管理线程。
2. 真实操作系统线程：thread.start() 后创建的后台执行流。
3. Worker 对象：你自己写的 QObject，通常通过 moveToThread(thread) 移到后台线程里执行任务。

最常见结构是：

self.thread = QThread()
self.worker = Worker()
self.worker.moveToThread(self.thread)
self.thread.started.connect(self.worker.run)
self.worker.finished.connect(self.thread.quit)
self.worker.finished.connect(self.worker.deleteLater)
self.thread.finished.connect(self.thread.deleteLater)
self.thread.start()

下面按生命周期讲。

⸻

1. QThread 本身不是“线程代码”

很多初学者会误解：

self.thread = QThread()

这句并没有立刻创建新线程。

它只是创建了一个 QThread 管理对象。

真正的新线程是在：

self.thread.start()

之后才启动。

启动后，Qt 会在新的操作系统线程里运行 QThread.run()。默认的 QThread.run() 会启动一个事件循环，大致等价于：

def run(self):
    self.exec()

也就是说，默认情况下，新线程里面会有一个 Qt 事件循环。

⸻

2. moveToThread() 是什么意思？

self.worker.moveToThread(self.thread)

这句的意思不是“马上执行 worker”，而是：

把 worker 的线程归属改成 self.thread 对应的后台线程。

更准确地说，QObject 有一个 thread affinity，也就是“属于哪个线程”。

当 worker 被 move 到后台线程后：

self.worker.thread() == self.thread

这意味着：

如果某个信号以队列连接方式调用 worker 的槽函数，那么这个槽函数会在 worker 所属的后台线程中执行。

⸻

3. thread.started.connect(worker.run) 的执行逻辑

self.thread.started.connect(self.worker.run)

当你调用：

self.thread.start()

Qt 启动后台线程，然后发射：

thread.started

因为 worker 已经被 moveToThread(self.thread)，所以 worker.run() 会在后台线程执行。

典型流程是：

主线程:
    创建 QThread
    创建 Worker
    worker.moveToThread(thread)
    thread.started.connect(worker.run)
    thread.start()
后台线程:
    QThread 启动
    发射 started 信号
    调用 worker.run()

⸻

4. worker 的 run() 一般长这样

例如：

class Worker(QObject):
    finished = pyqtSignal()
    progress = pyqtSignal(int)
    @pyqtSlot()
    def run(self):
        for i in range(100):
            time.sleep(0.05)
            self.progress.emit(i)
        self.finished.emit()

这里最重要的是最后：

self.finished.emit()

这个信号通常表示：

worker 的任务完成了，可以开始清理和退出线程了。

⸻

5. worker.finished.connect(thread.quit) 是退出线程的关键

self.worker.finished.connect(self.thread.quit)

这句的含义是：

当 worker 发出 finished 信号时，调用 thread.quit()。

而：

thread.quit()

的作用是：

请求该 QThread 的事件循环退出。

注意，quit() 不是强杀线程。

它不是：

立刻终止 CPU 执行

而是：

让线程事件循环结束

默认 QThread 的运行方式是：

def run(self):
    self.exec()

所以：

thread.quit()

本质上会让：

self.exec()

返回。

一旦 exec() 返回，QThread 的 run() 结束，后台线程自然结束。

流程如下：

worker.run() 完成
    ↓
worker.finished.emit()
    ↓
thread.quit()
    ↓
QThread 的事件循环退出
    ↓
QThread.run() 返回
    ↓
真实后台线程结束
    ↓
thread.finished 信号发出

⸻

6. worker.deleteLater() 是谁触发的？

这句：

self.worker.finished.connect(self.worker.deleteLater)

是由 worker 自己的 finished 信号触发的。

也就是说：

worker.finished.emit()
    ├── thread.quit()
    └── worker.deleteLater()

二者都是由 worker.finished 触发的。

不过要注意，deleteLater() 并不是马上删除对象。

它的意思是：

给 worker 所在线程的事件循环投递一个 DeferredDelete 事件，等事件循环有机会处理时再删除。

⸻

7. 那 worker 到底什么时候被删除？

关键点来了。

当你写：

self.worker.finished.connect(self.worker.deleteLater)
self.worker.finished.connect(self.thread.quit)

这两个槽函数的调用顺序通常和连接顺序有关，但不要把程序正确性建立在这个顺序上。

常见写法是：

self.worker.finished.connect(self.thread.quit)
self.worker.finished.connect(self.worker.deleteLater)

或者：

self.worker.finished.connect(self.worker.deleteLater)
self.worker.finished.connect(self.thread.quit)

理论上都能工作，但背后的细节是：

worker.deleteLater()

会向 worker 所在线程投递删除事件。

如果此时线程事件循环还在运行，它会处理这个删除事件，然后删除 worker。

如果事件循环马上 quit，Qt 通常也会在事件循环退出前处理 deferred delete 事件，但这类细节不建议你依赖过度。

更稳妥的理解是：

worker.finished.emit()
    ↓
安排 worker 稍后删除
    ↓
请求线程事件循环退出
    ↓
事件循环收尾
    ↓
worker 被删除
    ↓
线程结束

⸻

8. thread.finished.connect(thread.deleteLater) 是什么意思？

self.thread.finished.connect(self.thread.deleteLater)

这句表示：

当后台线程真正结束后，删除 QThread 管理对象。

注意，不是 thread.quit() 发出 finished。

而是：

线程事件循环退出
QThread.run() 返回
真实线程结束

之后，Qt 才会发射：

thread.finished

然后触发：

thread.deleteLater()

所以流程是：

worker.finished.emit()
    ↓
thread.quit()
    ↓
后台线程事件循环退出
    ↓
后台线程结束
    ↓
thread.finished.emit()
    ↓
thread.deleteLater()

⸻

9. 总流程图

完整流程大概是：

主线程
│
│ 创建 QThread
│ 创建 Worker
│ worker.moveToThread(thread)
│ 连接信号槽
│ thread.start()
│
└───────────────────────────────┐
                                │
后台线程启动
│
│ 发射 thread.started
│
│ 调用 worker.run()
│
│ worker 执行耗时任务
│
│ worker.finished.emit()
│
├── 调用 thread.quit()
│       ↓
│   请求退出后台线程事件循环
│
├── 调用 worker.deleteLater()
│       ↓
│   安排 worker 稍后删除
│
│ 后台线程事件循环退出
│
│ QThread.run() 返回
│
│ 真实线程结束
│
│ thread.finished.emit()
│
└── thread.deleteLater()
        ↓
    删除 QThread 管理对象

⸻

10. quit()、exit()、terminate() 的区别

quit()

thread.quit()

等价于：

thread.exit(0)

含义：

让线程事件循环以返回码 0 退出。

这是推荐方式。

⸻

exit(code)

thread.exit(1)

含义：

让线程事件循环退出，并返回指定 code。

一般 PyQt/PySide 里不太常用。

⸻

terminate()

thread.terminate()

含义：

强制终止线程。

这个很危险。

因为它可能在 worker 正在执行任何代码的时候强行打断，例如：

正在写文件
正在操作锁
正在释放资源
正在更新共享数据
正在调用 C/C++ 库

这会导致资源泄漏、死锁、数据损坏甚至程序崩溃。

所以一般不要用：

thread.terminate()

除非已经没有其他选择。

⸻

11. 为什么 quit() 有时不能让线程退出？

这是非常重要的问题。

thread.quit() 只能退出 线程事件循环。

但是如果你的 worker.run() 是一个长时间阻塞函数，例如：

def run(self):
    while True:
        time.sleep(1)

那么后台线程正在执行 worker.run()，事件循环没有机会继续处理。

这种情况下：

thread.quit()

可能不会立刻生效。

因为事件循环被你的 run() 占住了。

例如：

@pyqtSlot()
def run(self):
    while True:
        do_something()

这个函数一直不返回，那么线程事件循环就没有机会正常退出。

所以正确做法是让 worker 支持“主动停止”。

⸻

12. 正确的 worker 取消机制

典型写法：

class Worker(QObject):
    finished = pyqtSignal()
    def __init__(self):
        super().__init__()
        self._stop_requested = False
    @pyqtSlot()
    def run(self):
        while not self._stop_requested:
            self.do_one_step()
        self.finished.emit()
    @pyqtSlot()
    def stop(self):
        self._stop_requested = True

主线程中：

self.stop_button.clicked.connect(self.worker.stop)

或者关闭窗口时：

self.worker.stop()
self.thread.quit()
self.thread.wait()

更线程安全一点可以用：

from threading import Event
class Worker(QObject):
    finished = pyqtSignal()
    def __init__(self):
        super().__init__()
        self._stop_event = Event()
    @pyqtSlot()
    def run(self):
        while not self._stop_event.is_set():
            self.do_one_step()
        self.finished.emit()
    @pyqtSlot()
    def stop(self):
        self._stop_event.set()

核心原则是：

不要从外面强杀 worker，而是通知 worker 自己停下来。

⸻

13. 为什么要用 wait()？

在窗口关闭时，经常看到：

self.worker.stop()
self.thread.quit()
self.thread.wait()

其中：

self.thread.wait()

的意思是：

阻塞当前线程，直到后台线程真的结束。

通常在关闭程序时使用，确保后台线程彻底退出。

例如：

def closeEvent(self, event):
    if self.thread and self.thread.isRunning():
        self.worker.stop()
        self.thread.quit()
        self.thread.wait()
    event.accept()

否则可能出现程序退出时警告：

QThread: Destroyed while thread is still running

这个警告的意思是：

QThread 管理对象被销毁了，但它管理的真实线程还在运行。

这是不安全的。

⸻

14. deleteLater() 和 wait() 的关系

这个地方也容易混淆。

self.thread.finished.connect(self.thread.deleteLater)

是 Qt 的对象清理机制。

self.thread.wait()

是线程同步机制。

它们不是一回事。

操作	作用
thread.quit()	请求线程事件循环退出
thread.wait()	等待真实线程结束
thread.deleteLater()	稍后删除 QThread 对象
worker.deleteLater()	稍后删除 Worker 对象

通常完整关闭逻辑是：

worker.stop()
thread.quit()
thread.wait()

而对象清理靠：

worker.finished.connect(worker.deleteLater)
thread.finished.connect(thread.deleteLater)

⸻

15. 最推荐的基本模板

class Worker(QObject):
    finished = pyqtSignal()
    progress = pyqtSignal(int)
    def __init__(self):
        super().__init__()
        self._stop_requested = False
    @pyqtSlot()
    def run(self):
        try:
            for i in range(100):
                if self._stop_requested:
                    break
                time.sleep(0.05)
                self.progress.emit(i)
        finally:
            self.finished.emit()
    @pyqtSlot()
    def stop(self):
        self._stop_requested = True

主窗口：

class MainWindow(QMainWindow):
    def start_worker(self):
        self.thread = QThread(self)
        self.worker = Worker()
        self.worker.moveToThread(self.thread)
        self.thread.started.connect(self.worker.run)
        self.worker.progress.connect(self.on_progress)
        self.worker.finished.connect(self.thread.quit)
        self.worker.finished.connect(self.worker.deleteLater)
        self.thread.finished.connect(self.thread.deleteLater)
        self.thread.start()
    def closeEvent(self, event):
        if hasattr(self, "worker"):
            self.worker.stop()
        if hasattr(self, "thread") and self.thread.isRunning():
            self.thread.quit()
            self.thread.wait()
        event.accept()

⸻

16. finished.emit() 最好放在 finally 里

为什么？

假设 worker 中途出错：

def run(self):
    do_something()
    raise RuntimeError("出错了")
    self.finished.emit()

如果异常发生了，finished.emit() 就不会执行。

结果：

worker.finished.connect(thread.quit)

不会触发。

线程可能不退出。

所以更稳妥：

@pyqtSlot()
def run(self):
    try:
        do_something()
    except Exception as e:
        self.failed.emit(str(e))
    finally:
        self.finished.emit()

这样无论成功、失败还是取消，都会发出 finished，线程都会进入退出流程。

⸻

17. 常见错误一：直接操作 UI

错误写法：

def run(self):
    self.label.setText("完成")

如果 label 是主线程 UI 控件，而 worker 在后台线程，这就是跨线程操作 UI。

Qt 不允许这样做。

正确方式：

class Worker(QObject):
    message = pyqtSignal(str)
    def run(self):
        self.message.emit("完成")

主线程：

self.worker.message.connect(self.label.setText)

也就是说：

后台线程只发信号，主线程负责更新 UI。

⸻

18. 常见错误二：在 worker 还没结束时删除 thread

错误：

self.thread.start()
del self.thread

或者窗口关闭时没有等待线程退出。

可能导致：

QThread: Destroyed while thread is still running

正确做法：

self.worker.stop()
self.thread.quit()
self.thread.wait()

⸻

19. 常见错误三：以为 quit() 会停止 worker 的 while 循环

例如：

def run(self):
    while True:
        heavy_work()

然后你调用：

thread.quit()

这通常不能停掉循环。

因为循环本身还在跑。

应该写成：

def run(self):
    while not self._stop_requested:
        heavy_work()

然后：

worker.stop()

⸻

20. 常见错误四：worker 没有引用，被 Python 回收

如果你这样写：

def start_worker(self):
    thread = QThread()
    worker = Worker()
    worker.moveToThread(thread)
    thread.start()

函数结束后，thread 和 worker 可能被垃圾回收。

正确做法：

self.thread = QThread()
self.worker = Worker()

保存在实例属性上。

⸻

21. 最关键的一句话

Qt worker-thread 退出机制可以概括为：

Worker 完成任务后发出 finished，这个信号触发 thread.quit() 让线程事件循环退出，同时触发 worker.deleteLater() 安排 worker 删除；线程真正结束后发出 thread.finished，再触发 thread.deleteLater() 删除 QThread 管理对象。

也就是：

worker.finished
    ↓
thread.quit()
worker.deleteLater()
    ↓
thread event loop exits
    ↓
thread.finished
    ↓
thread.deleteLater()

但是要特别记住：

thread.quit() 只退出事件循环，不会强制打断正在运行的耗时函数。真正可靠的停止方式，是让 worker 自己检查取消标志，然后主动返回。
```

```text
对，你抓到关键点了：

worker.finished.connect(thread.quit)
worker.finished.connect(worker.deleteLater)

如果按连接顺序执行，那么确实是：

worker.finished.emit()
    ↓
先调用 thread.quit()
    ↓
再调用 worker.deleteLater()

所以你的疑问是对的：

既然 quit() 先执行了，后面的 deleteLater() 还有机会被处理吗？

答案是：通常还有机会，但不建议把正确性建立在这个细节上。

⸻

1. thread.quit() 不是“立刻把线程杀掉”

thread.quit() 的本质是：

thread.exit(0)

它做的是：

请求线程的事件循环退出。

它不是马上销毁线程，也不是马上让当前代码停止执行。

所以当 worker.finished.emit() 正在分发信号槽时，即使先调用了：

thread.quit()

后面的槽函数：

worker.deleteLater()

仍然会继续被调用。

也就是说，这个过程更接近：

worker.finished.emit()
    ↓
调用 thread.quit()，设置“事件循环该退出了”的状态
    ↓
继续调用 worker.deleteLater()
    ↓
finished 信号分发结束
    ↓
事件循环准备退出

不是：

worker.finished.emit()
    ↓
thread.quit()
    ↓
线程瞬间死亡
    ↓
worker.deleteLater() 没机会执行

后面这种理解是错的。

⸻

2. 但 deleteLater() 的删除动作不是立即发生

deleteLater() 不是直接 delete worker。

它是：

向 worker 所属线程的事件队列投递一个 DeferredDelete 事件。

也就是：

worker.deleteLater()
    ↓
给 worker 所在线程投递“稍后删除我”的事件

真正删除要等事件循环处理这个 DeferredDelete 事件。

所以你的担心进一步变成：

如果 quit() 已经让事件循环退出了，那 deleteLater() 投递的 DeferredDelete 事件还能不能被处理？

这个问题非常关键。

⸻

3. Qt 通常会在事件循环退出时处理 DeferredDelete

Qt 对 deleteLater() 有专门处理。即使事件循环准备退出，Qt 通常仍会处理 deferred delete 相关事件，避免这类常见写法泄漏对象。

所以这种写法在 Qt 里非常常见：

worker.finished.connect(thread.quit)
worker.finished.connect(worker.deleteLater)
thread.finished.connect(thread.deleteLater)

一般是可以工作的。

Qt 官方示例里也经常这么写，逻辑就是：

worker 完成
    ↓
请求线程退出
    ↓
安排 worker 删除
    ↓
线程收尾阶段处理 DeferredDelete
    ↓
worker 被销毁
    ↓
线程结束

⸻

4. 但更稳妥的连接顺序是：先 deleteLater，再 quit

如果你想让逻辑看起来更顺，完全可以写成：

worker.finished.connect(worker.deleteLater)
worker.finished.connect(thread.quit)

这样信号分发时的顺序就是：

worker.finished.emit()
    ↓
先调用 worker.deleteLater()
    ↓
投递 DeferredDelete 事件
    ↓
再调用 thread.quit()
    ↓
请求事件循环退出

从理解上更自然。

我个人更推荐你在教学代码或学习代码里这样写：

self.worker.finished.connect(self.worker.deleteLater)
self.worker.finished.connect(self.thread.quit)
self.thread.finished.connect(self.thread.deleteLater)

因为它表达得更清楚：

先安排 worker 删除，再退出线程事件循环

⸻

5. 但即使这样，也不要误解为 worker 立刻删除

即使你写：

worker.finished.connect(worker.deleteLater)
worker.finished.connect(thread.quit)

也不是：

deleteLater 立刻删除 worker

而是：

deleteLater 投递删除事件
quit 请求事件循环退出
事件循环处理 DeferredDelete
worker 才真正销毁

所以 deleteLater() 无论在 quit() 前还是后，本质都是“安排稍后删除”。

⸻

6. 更保险的方案：用 thread.finished 删除 worker？

有些人会写：

worker.finished.connect(thread.quit)
thread.finished.connect(worker.deleteLater)
thread.finished.connect(thread.deleteLater)

这个写法表面上看也合理：

worker 完成
    ↓
线程退出
    ↓
线程 finished 后删除 worker

但它有一个隐患：

worker.deleteLater() 应该在 worker 所属线程的事件循环还能处理删除事件时调用。

如果 thread.finished 已经发出了，说明后台线程事件循环已经结束了。此时再对属于这个线程的 worker 调用 deleteLater()，它可能没有合适的事件循环来处理 DeferredDelete。

所以不推荐把 worker 的 deleteLater() 连接到 thread.finished。

更推荐：

worker.finished.connect(worker.deleteLater)
worker.finished.connect(thread.quit)
thread.finished.connect(thread.deleteLater)

⸻

7. 关键区别：worker.deleteLater 和 thread.deleteLater 所在线程不同

这里还有一个细节。

worker 被 move 到后台线程：

worker.moveToThread(thread)

所以：

worker.deleteLater()

应该由后台线程的事件循环处理。

但 thread 这个 QThread 对象通常是主线程创建的，它本身仍属于主线程：

thread = QThread()

所以：

thread.deleteLater()

通常由主线程事件循环处理。

因此：

thread.finished.connect(thread.deleteLater)

是合理的，因为 QThread 管理对象本身不属于后台线程，而是属于创建它的主线程。

⸻

8. 推荐写法

我建议你使用这个顺序：

self.thread = QThread()
self.worker = Worker()
self.worker.moveToThread(self.thread)
self.thread.started.connect(self.worker.run)
self.worker.finished.connect(self.worker.deleteLater)
self.worker.finished.connect(self.thread.quit)
self.thread.finished.connect(self.thread.deleteLater)
self.thread.start()

逻辑顺序更清楚：

thread.start()
    ↓
后台线程事件循环启动
    ↓
thread.started
    ↓
worker.run()
    ↓
worker.finished.emit()
    ↓
worker.deleteLater()  安排 worker 删除
    ↓
thread.quit()         请求线程退出
    ↓
后台线程事件循环收尾
    ↓
worker 被真正删除
    ↓
线程结束
    ↓
thread.finished.emit()
    ↓
thread.deleteLater()  删除 QThread 管理对象

⸻

9. 你的理解可以修正成这样

你原来的理解：

quit 在前，deleteLater 在后，所以 deleteLater 没机会执行

需要改成：

quit 在前，只是请求事件循环退出；
当前 finished 信号的后续槽函数仍会继续执行；
deleteLater 仍然会被调用；
真正删除发生在事件循环处理 DeferredDelete 时。

但是从代码可读性和稳妥性上说，推荐写成：

worker.finished.connect(worker.deleteLater)
worker.finished.connect(thread.quit)

比：

worker.finished.connect(thread.quit)
worker.finished.connect(worker.deleteLater)

更容易理解。
```